In [10]:
#Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

# Step 2 Ml processing 
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, PolynomialFeatures, StandardScaler, MinMaxScaler  # for l
from sklearn.utils import shuffle
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
# from sklearn.preprocessing import PolynomialFeatures
# from sklearn.preprocessing import StandardScaler

### Task 1

Create Basic Interaction Features
Create interaction features between Age and Duration, and between Net Sales and Duration. Store the enhanced dataset in features_engineered.


In [11]:
# Load and prepare the travel insurance data
data = pd.read_csv('/home/susan/Feature_Engineering_Sprint_9/Data/travel_insurance_us.csv')
print(data.info())
print()
# Names of numeric columns
print(data.select_dtypes(include='number').columns.tolist())
# Basic preprocessing - handle missing values and extract numerical features
features= data.drop('Claim', axis= 1)
target = data['Claim']
X_train, X_valid, y_train, y_vaild = train_test_split (features, target, test_size= 0.25, random_state =12345 )

# Extract numerical features for feature engineering
numerical_features = [ 'Duration', 'Net Sales', 'Commission (in value)', 'Age']
features_numerical = X_train[numerical_features].copy()
# Your task: Create interaction features
features_engineered = features_numerical.copy()
# Create Age × Duration interaction
features_engineered['Age_Duration'] = features_engineered['Age'] * features_engineered ['Duration']

# Create Net Sales × Duration interaction
features_engineered['NetSales_Duration'] = features_engineered['Net Sales'] * features_engineered ['Duration']
print("Original features:", features_numerical.shape[1])
print()
print('=========='*20)
print("Engineered features:", features_engineered.shape[1])
print()
print('=========='*20)
print("\nNew feature columns:")
print(features_engineered.columns.tolist())
print("\nFirst 5 rows of engineered features:")
print(features_engineered.head())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50660 entries, 0 to 50659
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Agency                 50660 non-null  object 
 1   Agency Type            50660 non-null  object 
 2   Distribution Channel   50660 non-null  object 
 3   Product Name           50660 non-null  object 
 4   Claim                  50660 non-null  int64  
 5   Duration               50660 non-null  int64  
 6   Destination            50660 non-null  object 
 7   Net Sales              50660 non-null  float64
 8   Commission (in value)  50660 non-null  float64
 9   Gender                 14569 non-null  object 
 10  Age                    50660 non-null  int64  
dtypes: float64(2), int64(3), object(6)
memory usage: 4.3+ MB
None

['Claim', 'Duration', 'Net Sales', 'Commission (in value)', 'Age']
Original features: 4

Engineered features: 6


New feature columns:
['Duration',

### **Create Ratio Features**
 #### Task 2


Create meaningful ratio features that could provide business insights for insurance risk assessment.


In [12]:
#  Your task: Create ratio features
# Create Net Sales per Day (avoiding division by zero)

features_engineered['Sales_Per_Day'] = np.where(
    features_engineered['Duration'] != 0,
    features_engineered['Net Sales'] / features_engineered['Duration'],0)
# Create Commission Rate (commission as percentage of net sales)
# Handle cases where Net Sales might be 0 or negative
features_engineered['Commission_Rate'] =  np.where(features_engineered['Net Sales']>0,
                                                  features_engineered['Net Sales'] / features_engineered['Duration'],0)

print("Feature engineering complete!")
print("Total features:", features_engineered.shape[1])
print("\nFeature summary:")
print(features_engineered.describe())

Feature engineering complete!
Total features: 8

Feature summary:
           Duration     Net Sales  Commission (in value)           Age  \
count  37995.000000  37995.000000           37995.000000  37995.000000   
mean      49.806369     40.824758               9.815489     39.985840   
std      111.686404     48.385539              19.606478     14.065098   
min       -2.000000   -389.000000               0.000000      0.000000   
25%        9.000000     18.000000               0.000000     35.000000   
50%       22.000000     27.000000               0.000000     36.000000   
75%       53.000000     49.000000              11.550000     43.000000   
max     4881.000000    682.000000             262.760000    118.000000   

        Age_Duration  NetSales_Duration  Sales_Per_Day  Commission_Rate  
count   37995.000000       37995.000000   37995.000000     3.799500e+04  
mean     1997.240400        4137.024837       2.462495              inf  
std      5132.642505       15914.560528      

/home/susan/Feature_Engineering_Sprint_9/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


### **Create polynomial features**
#### Task 4 

Compare Feature Scales
Analyze the scale differences in your engineered features and understand why scaling is necessary.

In [13]:
#Create polynomial features
# What this does:

# Creates polynomial combinations of your 4 features
# Original features: Duration, Net Sales, Commission, Age
# New features: Duration², Net Sales², Age², Duration×Net Sales, Duration×Age, etc.
# Converts the result back to a DataFrame with proper column names
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(features_numerical)
feature_names = poly.get_feature_names_out(numerical_features)
poly_df = pd.DataFrame(poly_features, columns=feature_names, index=features_numerical.index)

# Your task: Analyze the scale differences
print("FEATURE SCALE ANALYSIS")
print("=" * 50)
# Calculate and display statistics for each feature
# feature_stats DataFrame from the .describe() method
feature_stats = poly_df.describe().T


# Identify the feature with the largest range
# Range = Maximum value - Minimum value (for each individual feature)
#Step 3: Find Features with Largest/Smallest Range Now we need two new methods:
# .idxmax() = "index of maximum" = tells you the NAME of the column with the biggest value
# .idxmin() = "index of minimum" = tells you the NAME of the column with the smallest value
#Access the rows directly
# Since 'max' and 'min' are row names, you'd need to use .loc[] to access them, like:
feature_stats['range'] = feature_stats['max'] - feature_stats['min']
max_range_feature = feature_stats['range'].idxmax()
min_range_feature = feature_stats['range'].idxmin()
print(feature_stats[['min', 'max', 'range']].round(2))
print(f"\nLargest range: {max_range_feature}")
print(f"Smallest range: {min_range_feature}")

print(f"\nScale difference ratio: {feature_stats['range'].max() / feature_stats['range'].min():.2f}:1")

FEATURE SCALE ANALYSIS
                                      min          max        range
Duration                             -2.0      4881.00      4883.00
Net Sales                          -389.0       682.00      1071.00
Commission (in value)                 0.0       262.76       262.76
Age                                   0.0       118.00       118.00
Duration^2                            0.0  23824161.00  23824161.00
Duration Net Sales              -141596.0    229417.00    371013.00
Duration Commission (in value)      -15.4    101101.00    101116.40
Duration Age                       -236.0    234288.00    234524.00
Net Sales^2                           0.0    465124.00    465124.00
Net Sales Commission (in value)  -52925.4    116281.00    169206.40
Net Sales Age                    -17884.8     49560.00     67444.80
Commission (in value)^2               0.0     69042.82     69042.82
Commission (in value) Age             0.0     19824.00     19824.00
Age^2                    

### **Apply StandardScaler**
#### Task 5

Apply StandardScaler to your polynomial features and observe how it transforms the data distributions.

In [14]:
# Basic preprocessing - handle missing values and extract numerical features
features= data.drop('Claim', axis= 1)
target = data['Claim']
X_train, X_valid, y_train, y_vaild = train_test_split (features, target, test_size= 0.25, random_state =12345 )

# Extract numerical features for feature engineering
numerical_features = [ 'Duration', 'Net Sales', 'Commission (in value)', 'Age']
poly = PolynomialFeatures(degree=2, include_bias = False)
# Fit polynomial features on training data only
poly_train = poly.fit_transform(X_train[numerical_features])
poly_valid = poly.transform(X_valid[numerical_features])
feature_names =poly.get_feature_names_out(numerical_features)
# Your task: Apply StandardScaler properly

scaler = StandardScaler()
scaler.fit(poly_train)  # Learn scaling parameters from training data only

poly_train_scaled = scaler.transform(poly_train)  # Apply to training data
poly_valid_scaled = scaler.transform(poly_valid)    # Apply same scaling to test data
# Convert back to DataFrames for analysis
train_scaled_df = pd.DataFrame(poly_train_scaled, columns=feature_names)
valid_scaled_df = pd.DataFrame(poly_valid_scaled, columns=feature_names)
print("BEFORE SCALING:")
print("Training data statistics:")
print(pd.DataFrame(poly_train, columns=feature_names).describe().loc[['mean', 'std']].round(3))

print("\nAFTER SCALING:")
print("Training data statistics:")
print(train_scaled_df.describe().loc[['mean', 'std']].round(3))

print(f"\nValidation data (first feature) - Mean: {valid_scaled_df.iloc[:, 0].mean():.3f}, Std: {valid_scaled_df.iloc[:, 0].std():.3f}")

BEFORE SCALING:
Training data statistics:
      Duration  Net Sales  Commission (in value)     Age  Duration^2  \
mean    49.806     40.825                  9.815  39.986   14954.199   
std    111.686     48.386                 19.606  14.065  403626.678   

      Duration Net Sales  Duration Commission (in value)  Duration Age  \
mean            4137.025                        1169.752      1997.240   
std            15914.561                        5409.455      5132.643   

      Net Sales^2  Net Sales Commission (in value)  Net Sales Age  \
mean     4007.760                         1012.882       1659.791   
std     13182.523                         4445.220       2232.654   

      Commission (in value)^2  Commission (in value) Age     Age^2  
mean                  480.748                    426.201  1796.689  
std                  2219.962                    906.351  1786.977  

AFTER SCALING:
Training data statistics:
      Duration  Net Sales  Commission (in value)  Age  Durati

### Task 6 

Compare Different Scaling Methods
Compare StandardScaler and MinMaxScaler on your travel insurance data to understand when each method is appropriate.

In [15]:
# Use just the original numerical features for clearer comparison
numerical_features = ['Duration', 'Net Sales', 'Commission (in value)', 'Age']
X_train = X_train[numerical_features].copy()
X_valid = X_valid[numerical_features].copy()

# Your task: Compare StandardScaler vs MinMaxScaler
print("ORIGINAL DATA RANGES:")
print("Min values:", X_train.min().values)
print("Max values:", X_train.max().values)
print()

# Apply StandardScaler
standard_scaler = StandardScaler()
standard_scaler.fit(X_train)
X_train_standard = standard_scaler.transform(X_train) 

# Apply MinMaxScaler
minmax_scaler = MinMaxScaler()
minmax_scaler.fit(X_train)
X_train_minmax = minmax_scaler.transform(X_train) 

# Convert to DataFrames for analysis
standard_df = pd.DataFrame(X_train_standard, columns=numerical_features)
minmax_df = pd.DataFrame(X_train_minmax, columns=numerical_features)

print("STANDARDSCALER RESULTS:")
print("Means:", standard_df.mean().round(3).values)
print("Stds: ", standard_df.std().round(3).values)
print("Mins: ", standard_df.min().round(3).values)
print("Maxs: ", standard_df.max().round(3).values)
print()

print("MINMAXSCALER RESULTS:")
print("Means:", minmax_df.mean().round(3).values)
print("Stds: ", minmax_df.std().round(3).values)
print("Mins: ", minmax_df.min().round(3).values)
print("Maxs: ", minmax_df.max().round(3).values)

ORIGINAL DATA RANGES:
Min values: [  -2. -389.    0.    0.]
Max values: [4881.    682.    262.76  118.  ]

STANDARDSCALER RESULTS:
Means: [ 0.  0. -0.  0.]
Stds:  [1. 1. 1. 1.]
Mins:  [-0.464 -8.883 -0.501 -2.843]
Maxs:  [43.257 13.252 12.901  5.547]

MINMAXSCALER RESULTS:
Means: [0.011 0.401 0.037 0.339]
Stds:  [0.023 0.045 0.075 0.119]
Mins:  [0. 0. 0. 0.]
Maxs:  [1. 1. 1. 1.]


### Feature Seclection
#### Task7 

Calculate Feature Correlations
Calculate correlations between your numerical features and the target variable to identify the most predictive features.

`Which features have the strongest relationship with Claim?`

#### Entire logic
Original data

      │
      ▼
Select numerical features

      │
      ▼
Engineer new features

      │
      ▼
Calculate correlation with Claim

      │
      ▼
Rank features

      │
      ▼
Keep only strong ones


In [16]:
# Step 1 : split data
train, valid = train_test_split(data, test_size=0.25, random_state=12345)

# Step 2: Keep only numerical features
numerical_features = ['Duration', 'Net Sales', 'Commission (in value)', 'Age']
X_train = train[numerical_features].copy()
y_train = train['Claim'].copy()
# Step 3: Create new engineered features

# Create some basic interaction features for analysis
X_train['Age_Duration'] = X_train['Age'] * X_train['Duration']
X_train['Sales_Per_Day'] = X_train['Net Sales'] / X_train['Duration']
#Step 4: Calculate correlation
 # task: Calculate correlations with target variable
correlations = X_train.corrwith(y_train)

print("FEATURE CORRELATIONS WITH INSURANCE CLAIMS:")
print("=" * 50)
# Display correlations sorted by absolute value
# Step 5: Sort them
abs_correlations = abs(correlations).sort_values(ascending=False)
for feature in abs_correlations.index:
    corr_value = correlations[feature]
    print("Feature: {:<20} Correlation: {:.4f}".format(feature, corr_value))
# Step 6: Choose a threshold
# Identify features with correlation above threshold
threshold = 0.05
important_features = correlations[abs(correlations) > threshold]

print("\nFEATURES ABOVE THRESHOLD ({:.2f}):".format(threshold))
print("Number of important features:", len(important_features))
for feature in important_features.index:
    print("- " + feature)


FEATURE CORRELATIONS WITH INSURANCE CLAIMS:
Feature: Net Sales            Correlation: 0.1355
Feature: Commission (in value) Correlation: 0.0982
Feature: Duration             Correlation: 0.0658
Feature: Age_Duration         Correlation: 0.0532
Feature: Age                  Correlation: -0.0150
Feature: Sales_Per_Day        Correlation: nan

FEATURES ABOVE THRESHOLD (0.05):
Number of important features: 4
- Duration
- Net Sales
- Commission (in value)
- Age_Duration


/home/susan/Feature_Engineering_Sprint_9/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:132: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


### Task 8

Select Features Using Random Forest
Use Random Forest feature importance to identify the most useful features for predicting insurance claims.

In [17]:
# Step 1 split data

train , valid = train_test_split(data, test_size= 0.25, random_state= 12345)

# print numeric features column name
numerical_features_name =data.select_dtypes(include='number').columns.tolist()
print(numerical_features_name)
numerical_features= [ 'Duration', 'Net Sales', 'Commission (in value)', 'Age']

# Create features and target
X_train = train[numerical_features].copy()
y_train = train['Claim'].copy()

# Feature Engineering or Add simple interaction features 
X_train['Age_Duration'] = X_train['Age']* X_train['Duration']
X_train['Age_Sales'] = X_train['Age'] * X_train['Net Sales']

# Train Random Forest and get feature importance
rf_model = RandomForestClassifier(random_state= 12345)
rf_model.fit(X_train, y_train) 
# Get feature importances
feature_imprtances = rf_model.feature_importances_
print()
print("RANDOM FOREST FEATURE IMPORTANCE:")
print("=" * 40)
print(feature_imprtances)

# create a dataFrame
importance_df = (pd.DataFrame({'feature':X_train.columns,
                              'importance':feature_imprtances })
                                .sort_values('importance',ascending= False))
# Display results
# Display results
for i in range(len(importance_df)):
    feature = importance_df.iloc[i]['feature']
    importance = importance_df.iloc[i]['importance']
    print(f"{i+1}: {feature:<20} Importance: {importance:.4f}")
#   print("{}: {:<20} Importance: {:.4f}".format(i+1, feature, importance))
 # Select top 3 features
top_features =importance_df.head(3)['feature'].tolist()
print("\nTOP 3 MOST IMPORTANT FEATURES:")
for feature in top_features:
    print("- " + feature)



['Claim', 'Duration', 'Net Sales', 'Commission (in value)', 'Age']

RANDOM FOREST FEATURE IMPORTANCE:
[0.20066294 0.13186698 0.10329691 0.11205094 0.25148412 0.20063812]
1: Age_Duration         Importance: 0.2515
2: Duration             Importance: 0.2007
3: Age_Sales            Importance: 0.2006
4: Net Sales            Importance: 0.1319
5: Age                  Importance: 0.1121
6: Commission (in value) Importance: 0.1033

TOP 3 MOST IMPORTANT FEATURES:
- Age_Duration
- Duration
- Age_Sales


I have a dataframe which I need to visit each row (len())
get features get imprtance and print them 


### Task 9 

Compare Models with Different Feature Sets
Compare the performance of models trained with all features versus models trained with only the most important features.


In [18]:
# split data 
train, valid = train_test_split(data, test_size= 0.25 , random_state= 12345)

print(train.select_dtypes(include='number').columns.tolist())
 
# prepare features and traget for both train and validation 
numerical_features= ['Duration', 'Net Sales', 'Commission (in value)', 'Age']

X_train = train[numerical_features].copy()
X_valid = valid[numerical_features].copy()
y_train = train['Claim'].copy()
y1_valid = valid['Claim'].copy()

# add simple interaction features to both sets
for df in [X_train, X_valid]:
    df['Ag_Duration'] = df['Age'] * df['Duration']
    df['Ag_Sales'] = df['Age'] * df['Net Sales']

# Train model with ALL features
print("TRAINING WITH ALL FEATURES:")
model_all = RandomForestClassifier(random_state= 12345)
model_all.fit(X_train, y_train) 
#So, in short: X_train = questions, y_train = correct answers, 
# and fit = learning from both.

# Make predictions and calculate accuracy
# X_valid and y_valid are used to check model performance on unseen data during development.
# X_valid: features for validation samples
# y_valid: true labels for those same
pred_all = model_all.predict(X_valid)  # model predictions
accuracy_all = accuracy_score(y1_valid, pred_all) # model predictions
print("Number of features used: {}".format(X_train.shape[1]))
print("Validation accuracy: {:.4f}".format(accuracy_all))

# Your task: Train model with only top 3 features
# First, identify top 3 features using Random Forest importance
rf_selector = RandomForestClassifier(random_state=12345)
rf_selector.fit(X_train , y_train)
feature_importances1 = rf_selector.feature_importances_

print(feature_importances1)

# covert to dataFrame
importance_df1 =(pd.DataFrame({'features': X_train.columns,
                               'importance1':feature_importances1})
                               .sort_values('importance1', ascending= False))

#you need to add ['feature'].tolist() to get the actual feature names:

top_3_feature = importance_df1.head(3)['features'].tolist()

# Train model with selected features only and traget stay the same and donot need to change them
print("\nTRAINING WITH TOP 3 FEATURES:")
X_train_selected =X_train[top_3_feature]
X_valid_selected = X_valid[top_3_feature]

model_selected = RandomForestClassifier(random_state=12345)
model_selected.fit(X_train_selected, y_train)
# Make predictions and calculate accuracy
pred_selected = model_selected.predict(X_valid_selected)
accuracy_selected = accuracy_score(y1_valid, pred_selected)
print("Features used: {}".format(', '.join(top_3_feature)))
print("Number of features used: {}".format(len(top_3_feature)))
print("Validation accuracy: {:.4f}".format(accuracy_selected))
print("\nCOMPARISON:")
print(f"All features accuracy:{accuracy_all:.4f}")
print("Top 3 features accuracy: {:.4f}".format(accuracy_selected))

['Claim', 'Duration', 'Net Sales', 'Commission (in value)', 'Age']
TRAINING WITH ALL FEATURES:
Number of features used: 6
Validation accuracy: 0.9848
[0.20066294 0.13186698 0.10329691 0.11205094 0.25148412 0.20063812]

TRAINING WITH TOP 3 FEATURES:
Features used: Ag_Duration, Duration, Ag_Sales
Number of features used: 3
Validation accuracy: 0.9852

COMPARISON:
All features accuracy:0.9848
Top 3 features accuracy: 0.9852
